# Recorrência temporal: taxa de reciclagem (Bloco 2)

**Pergunta de produto:** alegações falsas voltam a circular depois de checadas? Se sim, um sistema que *recupera a checagem antiga* quando a alegação reaparece tem valor. Se quase nada volta, o produto precisa de outro enquadramento (é o go/no-go do Bloco 3).

**O que vamos construir, em uma frase:** para cada alegação *i*, procurar no corpus uma alegação *j* que (1) diz a mesma coisa (similaridade ≥ τ) e (2) foi checada **antes**, pelo menos N dias antes. A fração de alegações que têm esse "gêmeo mais velho" é a taxa de reciclagem.

Cada seção abaixo tem:
- 🧠 **Aposte antes de rodar**: escreva seu palpite. Errar o palpite é o que faz aprender.
- 👀 **O que observar**: como ler o número que apareceu.

In [1]:
%pip install -q pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
SEED = 42

## 1. Carregar e alinhar

Três fontes precisam estar **alinhadas linha a linha**: o CSV, a matriz de embeddings (linha *k* = vetor da alegação *k*) e os clusters do 02. Se desalinhar, todo número depois fica errado sem dar erro. Por isso guardamos `idx` (a linha original no `com_texto_limpo.csv`) e fazemos `assert`.

Saem do corpus:
- **multi-alegação** ("Veja o que é #FATO ou #FAKE…"): não são uma alegação só.
- **sem data** (`NaT`): sem data não existe "antes/depois".

In [3]:
df = pd.read_csv("data/com_texto_limpo.csv")
embeddings = np.load("data/emb_minilm.npy")
assert len(df) == len(embeddings)

df["data"] = pd.to_datetime(df["Data da checagem"])
df["idx"] = np.arange(len(df))

manter = (~df["multi_claim"] & df["data"].notna()).to_numpy()
df = df[manter].reset_index(drop=True)
E = embeddings[manter]

clusters = pd.read_csv("data/clusters.csv")[["idx", "cluster_hdbscan"]]
df = df.merge(clusters, on="idx", how="left", validate="one_to_one")
assert len(df) == len(E)

print(f"Alegações: {len(df)} | período: {df['data'].min().date()} a {df['data'].max().date()} "
      f"({(df['data'].max() - df['data'].min()).days} dias)")

Alegações: 1850 | período: 2022-08-01 a 2022-12-01 (122 dias)


👀 O período tem ~4 meses. Então N (a distância mínima em dias) precisa caber nisso: N = 180 é impossível e N = 60 só aceita pares de agosto × outubro/novembro. Guarde isso para as ressalvas do `design.md`.

## 2. A matriz de similaridade S

Os embeddings foram gerados com `normalize_embeddings=True`: todo vetor tem norma 1. Com norma 1, o cosseno entre *a* e *b* é só o produto escalar `a · b`. Então `E @ E.T` calcula **todas** as similaridades de uma vez: `S[i, j] = cos(i, j)`.

Isso é o k-NN **exato** do design.md: comparar tudo com tudo. Com ~1.850 alegações são ~3,4 milhões de números (~14 MB em float32). Com 10 milhões de alegações não caberia, e aí entra busca aproximada (FAISS/HNSW). Para o experimento, o exato é melhor porque não adiciona erro.

🧠 **Aposte:** qual o valor da diagonal `S[i, i]`? A matriz é simétrica?

In [ ]:
S = E @ E.T
print("shape:", S.shape, "| memória:", round(S.nbytes / 1e6, 1), "MB")
print("diagonal ≈ 1?", np.allclose(np.diag(S), 1, atol=1e-4))
print("simétrica?", np.allclose(S, S.T, atol=1e-6))

# A diagonal é a alegação com ela mesma: tiramos do jogo, senão todo mundo é "gêmeo" de si
np.fill_diagonal(S, -1)

Uma linha de S é "a alegação *i* contra o corpus inteiro". Ordenar essa linha **é** a busca semântica que o produto vai fazer.

In [ ]:
def vizinhos(i, k=5):
    print(f"ALVO [{i}] ({df.at[i, 'data'].date()}, {df.at[i, 'Agência']}): {df.at[i, 'claim']}")
    for j in np.argsort(-S[i])[:k]:
        print(f"  {S[i, j]:.3f} | {df.at[j, 'data'].date()} | {df.at[j, 'Agência']:<20.20} | {df.at[j, 'claim'][:90]}")

vizinhos(df.index[df["claim"].str.contains("agronegócio", na=False)][0])

## 3. Calibrar a intuição sobre τ

τ é o limiar: acima dele, "é a mesma alegação". O modelo não sabe o que é "mesma alegação". Ele só dá um número. **Quem dá significado ao número é você**, olhando exemplos.

Primeiro, a distribuição: para cada alegação, qual a similaridade do vizinho mais próximo?

🧠 **Aposte:** que fração das alegações tem algum vizinho com similaridade ≥ 0,90?

In [ ]:
nn = S.max(axis=1)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(nn, bins=50, color="steelblue")
for t in [0.75, 0.80, 0.85, 0.90]:
    ax.axvline(t, color="crimson", ls="--", lw=1)
ax.set_xlabel("similaridade com o vizinho mais próximo"); ax.set_ylabel("nº de alegações")
plt.show()
for t in [0.75, 0.80, 0.85, 0.90, 0.95]:
    print(f"vizinho ≥ {t:.2f}: {(nn >= t).mean():.1%}")

Agora pares reais por **faixa** de similaridade. Leia cada faixa e decida, em voz alta: é a mesma alegação, o mesmo tema ou outra coisa?

In [ ]:
rng = np.random.default_rng(SEED)
iu, ju = np.triu_indices(len(df), k=1)  # cada par uma vez (i < j)
s_pares = S[iu, ju]

for lo, hi in [(0.70, 0.75), (0.75, 0.80), (0.80, 0.85), (0.85, 0.90), (0.90, 0.95), (0.95, 1.01)]:
    na_faixa = np.flatnonzero((s_pares >= lo) & (s_pares < hi))
    print(f"\n=== faixa [{lo:.2f}, {hi:.2f}): {len(na_faixa)} pares ===")
    for k in rng.choice(na_faixa, size=min(4, len(na_faixa)), replace=False):
        i, j = iu[k], ju[k]
        print(f"  {s_pares[k]:.2f} | {df.at[i, 'claim'][:70]}\n       | {df.at[j, 'claim'][:70]}")

👀 Nas faixas baixas aparecem pares do **mesmo tema** ("Lula" + "Pix" vs "Lula" + "2º turno"). Nas altas, a mesma alegação com palavras diferentes, ou a mesma *narrativa* com detalhe trocado (a mesma montagem de pesquisa, com outra data). Essa segunda é exatamente o tipo de reciclagem que o produto quer pegar.

## 4. A dimensão tempo

Queremos `data_i − data_j` para **todos** os pares. Em vez de dois `for`, usamos *broadcasting*: `a[:, None]` vira uma coluna (n×1), `a[None, :]` vira uma linha (1×n), e a subtração gera a matriz n×n. Veja num exemplo de 3 elementos:

In [ ]:
a = np.array([10, 20, 35])
print(a[:, None] - a[None, :])   # linha i, coluna j = a[i] - a[j]

**Decisão de definição (importante):** usamos a diferença **com sinal**, `delta[i, j] = data_i − data_j`. `delta ≥ N` significa "*j* foi checada pelo menos N dias **antes** de *i*". Então *i* é a **reciclagem** e *j* é o original.

A alternativa, `|delta| ≥ N`, conta os dois lados do par: o original também seria marcado como "reciclado". Isso infla a taxa (quase dobra) e não corresponde à pergunta de produto: "quando algo chega, já foi checado antes?".

In [ ]:
datas = df["data"].to_numpy()
delta = (datas[:, None] - datas[None, :]) / np.timedelta64(1, "D")  # em dias, com sinal
print("delta[0, :5] =", delta[0, :5])

## 5. Os filtros, um de cada vez

Nem todo par "similar e distante no tempo" é reciclagem. Há duas armadilhas conhecidas:

1. **Agregador:** o *Fato ou Boato* (Justiça Eleitoral) republica checagens de parceiros, dias depois. O par (checagem original, republicação) parece reciclagem, mas é a mesma checagem copiada. Tiramos todo par que envolva o agregador.
2. **Duplicata:** textos quase idênticos (`cos > 0,98`) da **mesma agência** tendem a ser a mesma matéria atualizada ou republicada, e não a alegação voltando. As duplicatas exatas de coleta já saíram no 00.

E uma armadilha que o **N resolve sozinho:** várias agências checando a mesma alegação na mesma semana é *cobertura paralela*, não reciclagem. Por isso N nunca é 0.

A função abaixo junta tudo. Tudo são matrizes booleanas n×n combinadas com `&` (e) e `~` (não).

In [ ]:
agencia = df["Agência"].to_numpy(dtype=object)  # pandas 3: .to_numpy() evita comparações estranhas com o tipo string
agregador = df["agregador"].to_numpy(dtype=bool)

mesma_agencia = agencia[:, None] == agencia[None, :]
envolve_agregador = agregador[:, None] | agregador[None, :]

def pares_reciclagem(tau, N, tirar_agregador=True, tirar_duplicata=True):
    """Matriz booleana: P[i, j] = True se j é um 'original' de i (similar e ≥ N dias antes)."""
    P = (S >= tau) & (delta >= N)
    if tirar_agregador:
        P &= ~envolve_agregador
    if tirar_duplicata:
        P &= ~((S > 0.98) & mesma_agencia)
    return P

def taxa_reciclagem(tau, N, **filtros):
    # .any(axis=1): a linha i tem PELO MENOS UM original?
    return pares_reciclagem(tau, N, **filtros).any(axis=1).mean()

🧠 **Aposte:** qual filtro remove mais pares, o N ou o agregador? Rode o funil para ver o efeito de cada peça (τ = 0,85 fixo, só como exemplo).

In [ ]:
tau_ex = 0.85
funil = {
    "similar (S ≥ τ), qualquer data":        ((S >= tau_ex) & (delta >= 0)).sum(),
    "+ j pelo menos 7 dias antes":           pares_reciclagem(tau_ex, 7, tirar_agregador=False, tirar_duplicata=False).sum(),
    "+ sem agregador":                       pares_reciclagem(tau_ex, 7, tirar_duplicata=False).sum(),
    "+ sem duplicata da mesma agência":      pares_reciclagem(tau_ex, 7).sum(),
}
pd.Series(funil, name="nº de pares")

## 6. ✍️ Critério go/no-go: escreva ANTES da próxima célula

Este é o passo que mais protege contra autoengano: se você olhar a taxa primeiro, vai (sem querer) escolher o critério que dá o resultado que você quer.

Preencha aqui e **copie para o `design.md` já**, com a data:

- **GO se:** taxa ≥ ___ % no τ validado, com precisão manual ≥ ___ % e N = ___ dias.
- **NO-GO se:** …
- **Por que esses números:** (ex.: abaixo de X%, o sistema raramente teria o que recuperar; precisão abaixo de Y%, o checador perde confiança)

## 7. A grade τ × N

Agora sim. Cada célula é a taxa de reciclagem para um par (τ, N).

In [ ]:
TAUS = [0.75, 0.80, 0.85, 0.90]
NS = [7, 14, 30, 60]

grade = pd.DataFrame([[taxa_reciclagem(t, n) for n in NS] for t in TAUS],
                     index=pd.Index(TAUS, name="τ"), columns=pd.Index(NS, name="N (dias)"))
grade.to_csv(RESULTS / "taxa_reciclagem_grade.csv")

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(grade.values * 100, cmap="Blues")
ax.set_xticks(range(len(NS)), NS); ax.set_yticks(range(len(TAUS)), TAUS)
ax.set_xlabel("N (dias)"); ax.set_ylabel("τ")
for a in range(len(TAUS)):
    for b in range(len(NS)):
        ax.text(b, a, f"{grade.values[a, b]:.1%}", ha="center", va="center",
                color="white" if grade.values[a, b] > grade.values.max() / 2 else "black")
ax.set_title("Taxa de reciclagem (% das alegações)")
fig.savefig(RESULTS / "taxa_reciclagem_grade.png", dpi=150, bbox_inches="tight")
plt.show()

👀 **O que observar:**
- A taxa **cai** quando τ sobe (exige mais parecido) e quando N sobe (exige mais distante). Se não cair, há bug.
- Veja o quanto a taxa muda entre τ vizinhos. Se ela varia muito, **a decisão depende de escolher bem o τ**, e o τ não pode ser escolhido "no olho". Isso é o que a próxima seção resolve.

## 8. Validação manual: medir a precisão do detector

A grade diz *quantos* pares o detector encontra, mas não se eles são reciclagem de verdade. A **precisão** responde: dos pares que o detector aponta, que fração é realmente a mesma alegação?

Amostramos pares em cada faixa de similaridade (com N = 7 e os filtros) e **você rotula**. A célula cria `results/validacao_pares.csv`. Abra o arquivo, preencha a coluna `rotulo` com:
- `mesma`: mesma alegação (mesmo fato afirmado, mesmo que com outras palavras ou outro detalhe)
- `tema`: mesmo assunto, afirmação diferente
- `diferente`: não tem relação

A célula não sobrescreve o arquivo se ele já existir, então seus rótulos ficam salvos.

In [ ]:
N_VAL = 7
FAIXAS = [(0.75, 0.80), (0.80, 0.85), (0.85, 0.90), (0.90, 1.01)]
POR_FAIXA = 8

caminho_val = RESULTS / "validacao_pares.csv"
if not caminho_val.exists():
    P = pares_reciclagem(0.75, N_VAL)
    ii, jj = np.nonzero(P)
    sims = S[ii, jj]
    linhas = []
    for lo, hi in FAIXAS:
        na_faixa = np.flatnonzero((sims >= lo) & (sims < hi))
        for k in rng.choice(na_faixa, size=min(POR_FAIXA, len(na_faixa)), replace=False):
            i, j = ii[k], jj[k]
            linhas.append({"faixa": f"{lo:.2f}-{min(hi, 1):.2f}", "sim": round(float(sims[k]), 3),
                           "dias": int(delta[i, j]), "idx_i": df.at[i, "idx"], "idx_j": df.at[j, "idx"],
                           "reciclada_i": df.at[i, "claim"], "original_j": df.at[j, "claim"], "rotulo": ""})
    pd.DataFrame(linhas).to_csv(caminho_val, index=False)
    print(f"Criado {caminho_val} com {len(linhas)} pares: preencha a coluna 'rotulo'.")

validacao = pd.read_csv(caminho_val)
validacao[["faixa", "sim", "dias", "reciclada_i", "original_j", "rotulo"]]

### Precisão por faixa → escolha do τ

Depois de rotular, rode de novo. A precisão **acumulada a partir do τ** ("se eu cortar em τ, que fração do que passa é mesma alegação?") é o que importa para escolher o τ: o menor τ cuja precisão acumulada bate o seu critério.

In [ ]:
rotulados = validacao[validacao["rotulo"].isin(["mesma", "tema", "diferente"])]
if len(rotulados) < len(validacao):
    print(f"⚠️ {len(validacao) - len(rotulados)} pares sem rótulo: preencha antes de escolher o τ.")

if len(rotulados):
    rotulados = rotulados.assign(acerto=rotulados["rotulo"].eq("mesma"))
    por_faixa = rotulados.groupby("faixa")["acerto"].agg(n="size", precisao="mean")
    por_faixa["precisao_acumulada (≥ faixa)"] = [
        rotulados.loc[rotulados["sim"] >= float(f.split("-")[0]), "acerto"].mean() for f in por_faixa.index]
    por_faixa.to_csv(RESULTS / "precisao_por_faixa.csv")
    display(por_faixa.round(2))

👀 Com 8 pares por faixa, cada par vale 12,5 pontos de precisão: o número é **ruidoso**. Registre o n junto da precisão no `design.md`. Se a decisão cair na borda do critério, rotule mais pares daquela faixa (aumente `POR_FAIXA`, apague o CSV e rotule de novo) antes de decidir.

**Defina o τ e o N escolhidos:**

In [ ]:
TAU = 0.85   # ← troque pelo τ que a validação justificou
N = 7        # ← e pelo N do seu critério
print(f"Taxa de reciclagem com τ={TAU}, N={N}: {taxa_reciclagem(TAU, N):.1%}")

## 9. Quais temas reciclam mais?

Liga com o Bloco 1: a taxa por cluster mostra *onde* a reciclagem se concentra. Isso ajuda a escolher o conjunto de teste (Bloco 4) e a explicar o resultado (ex.: "fraude nas urnas" volta sempre; boatos de evento pontual, não).

In [ ]:
df["reciclada"] = pares_reciclagem(TAU, N).any(axis=1)

por_cluster = (df[df["cluster_hdbscan"] >= 0]
               .groupby("cluster_hdbscan")
               .agg(tamanho=("reciclada", "size"), taxa=("reciclada", "mean"),
                    exemplo=("claim", "first"))
               .query("tamanho >= 15")
               .sort_values("taxa", ascending=False))
por_cluster.to_csv(RESULTS / "reciclagem_por_cluster.csv")
por_cluster.head(10).style.format({"taxa": "{:.1%}"})

## 10. Guardar os pares para o conjunto de teste

Os pares rotulados como `mesma` viram a categoria **temporal** do conjunto de teste (Bloco 4): a query é a alegação reciclada (*i*) e o alvo correto é o original (*j*). É o caso mais realista do teste, porque foi a própria desinformação que reescreveu a alegação, e não nós.

In [ ]:
if len(rotulados):
    temporais = rotulados[rotulados["acerto"]][["idx_i", "idx_j", "sim", "dias", "reciclada_i", "original_j"]]
    temporais.to_csv("data/pares_temporais.csv", index=False)
    print(f"{len(temporais)} pares temporais salvos em data/pares_temporais.csv")

## Checklist antes do Bloco 3

- [ ] Critério go/no-go escrito no `design.md` **antes** de ver a grade
- [ ] `validacao_pares.csv` rotulado; precisão por faixa com n
- [ ] τ e N escolhidos com justificativa (a precisão), e a taxa final
- [ ] Ressalvas: o corpus cobre ~4 meses (um ciclo eleitoral); há um só modelo de embedding; o n da validação é pequeno